Question Answering over Documents

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

In [4]:
from langchain_classic.chains import RetrievalQA
from langchain_ollama import ChatOllama
from langchain_classic.document_loaders import CSVLoader
from langchain_community.vectorstores import DocArrayInMemorySearch
from IPython.display import display, Markdown

file = "data/OutdoorClothingCatalog_1000.csv"
loader = CSVLoader(file_path=file, encoding="UTF-8")

In [11]:
llm = ChatOllama(model=os.getenv('MODEL_NAME'), temperature=0)

直接使用向量存储查询

In [6]:
# 基于文档加载器创建向量存储 vector store
from langchain_classic.indexes import VectorstoreIndexCreator
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings()

index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,  # 指定向量存储类
    embedding=embeddings
).from_loaders([loader])    # 从加载器中调用，接收一个文档加载器列表

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
# 查询
query = "Please list all your shirts with sun protection in a table in markdown and summarize each one."

response = index.query(
    question=query,   # 使用索引查询创建一个响应，并传入这个查询
    llm=llm
)

display(Markdown(response)) # 查看查询返回的内容

Here is a summary of the shirts with sun protection listed in a table:

| Model Name | Sun Protection Features | Fabric & Care | Additional Features |
|------------|------------------------|---------------|--------------------|
| Women's Tropical Tee, Sleeveless | - UPF 50+<br>- Built-in SunSmart™ protection<br>- Wrinkle resistant<br>- Low-profile pockets and side shaping for a flattering fit | - Shell: 71% nylon, 29% polyester<br>- Cape lining: 100% polyester<br>- Machine wash and dry | - Updated design with smoother buttons<br>- Front and back cape venting<br>- Two front pockets, tool tabs, and eyewear loop |
| Sun Shield Shirt by | - UPF 50+<br>- Wicks moisture for quick-drying comfort<br>- Fits comfortably over swimsuits<br>- Abrasion resistant | - 78% nylon, 22% Lycra Xtra Life fiber<br>- Handwash, line dry | - Imported |
| Men's Plaid Tropic Shirt, Short-Sleeve | - UPF 50+<br>- SunSmart technology blocks 98% of harmful UV rays<br>- High-performance fabric is wrinkle-free and quickly evaporates perspiration | - 52% polyester, 48% nylon<br>- Machine washable and dryable | - Front and back cape venting<br>- Two front bellows pockets |
| Tropical Breeze Shirt | - UPF 50+<br>- Wrinkle-resistant and moisture-wicking fabric keeps you cool and comfortable | - Shell: 71% nylon, 29% polyester<br>- Cape lining: 100% polyester<br>- Polyester-mesh inserts<br>- Machine wash and dry | - Traditional fit for a relaxed look<br>- UPF 50+ coverage<br>- Innovative SunSmart technology blocks 98% of harmful UV rays |

All these shirts offer high levels of sun protection (UPF 50+) with various additional features such as moisture-wicking, quick-drying, and comfortable fits.

创建检索问答链回答问题(相当于把上面的步骤细化)

In [15]:
# embedding models
embeddings = HuggingFaceEmbeddings()

embed = embeddings.embed_query("hello, world.")

print(f"{len(embed)=}")
print(f"{embed[:5]=}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


len(embed)=768
embed[:5]=[0.03653831407427788, 0.01959453895688057, -0.011175376363098621, -0.03309578076004982, 0.05406976491212845]


In [16]:
# 创建并查询vector database
db = DocArrayInMemorySearch.from_documents(
    documents=loader.load(),
    embedding=embeddings,
)

query = "Please suggest a shirt with sunblocking"

answer = db.similarity_search(query=query)

print(f"返回文档数量： \n{len(answer)=} \n第一个文档：\n{answer[0]}")

返回文档数量： 
len(answer)=4 
第一个文档：
page_content=': 255
name: Sun Shield Shirt by
description: "Block the sun, not the fun – our high-performance sun shirt is guaranteed to protect from harmful UV rays. 

Size & Fit: Slightly Fitted: Softly shapes the body. Falls at hip.

Fabric & Care: 78% nylon, 22% Lycra Xtra Life fiber. UPF 50+ rated – the highest rated sun protection possible. Handwash, line dry.

Additional Features: Wicks moisture for quick-drying comfort. Fits comfortably over your favorite swimsuit. Abrasion resistant for season after season of wear. Imported.

Sun Protection That Won't Wear Off
Our high-performance fabric provides SPF 50+ sun protection, blocking 98% of the sun's harmful rays. This fabric is recommended by The Skin Cancer Foundation as an effective UV protectant.' metadata={'source': 'data/OutdoorClothingCatalog_1000.csv', 'row': 255}


In [20]:
# 基于database创建retriever
retriever = db.as_retriever()

qa_stuff = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    verbose=True,
)

query = "Please list all your shirts with sun protection in a table in markdown and summarize each one."

response = qa_stuff.invoke(input=query)

display(Markdown(response["result"]))



> Entering new RetrievalQA chain...

> Finished chain.


Here is the information about the shirts with sun protection listed in a table format:

|Shirt Name| Description|
|---|---|
|Women's Tropical Tee, Sleeveless| A five-star sleeveless button-up shirt that provides SunSmart™ protection to block harmful UV rays. It has an updated design with smoother buttons and is wrinkle-resistant.|
|Sunrise Tee| A lightweight, high-performance button-down shirt designed for hot weather. It offers UPF 50+ sun protection and moisture-wicking properties. The fabric resists wrinkles and dries quickly.|
|Sun Shield Shirt| A high-performance sun shirt that guarantees protection from harmful UV rays. It is made of a blend of nylon and Lycra Xtra Life fiber, providing UPF 50+ rating. It can be handwashed and line dried.|
|Tropical Breeze Shirt (Men's)| A lightweight, breathable long-sleeve UPF shirt that offers superior SunSmart™ protection from the sun’s harmful rays. The fabric is wrinkle-resistant and moisture-wicking, keeping you cool and comfortable.| 

Each of these shirts provides high levels of sun protection with ratings ranging from 50+ to SPF 50+. They are made from materials designed for comfort, quick-drying, and resistance to wrinkles. Some also have additional features such as front and back cape venting or built-in pockets.